# Software FJ | Gestión de reservas
Curso de Programación 213023 · Fase 4 · UNAD

Ejecute las celdas en orden con Python 3. Los datos son ficticios y los valores están en pesos colombianos. Al repetir la simulación se reinician las listas y se reemplaza el registro de la ejecución anterior.

## 1. Validaciones compartidas
La validación se concentra en tres funciones. Al convertir una tarifa ilegible, raise … from conserva la causa original para investigarla en el registro.

In [1]:
from abc import ABC, abstractmethod
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path
import logging
import re


class ErrorGestion(Exception):
    """Error recuperable que puede explicarse al usuario."""


class DatoInvalido(ErrorGestion):
    """Un dato incumple una regla de entrada."""


class OperacionInvalida(ErrorGestion):
    """El recurso o su estado impide la operación."""


def texto(valor, campo):
    if not isinstance(valor, str) or not valor.strip():
        raise DatoInvalido(f"Falta un valor de texto para {campo}.")
    return valor.strip()


def entero(valor, campo):
    if type(valor) is not int or valor < 1:
        raise DatoInvalido(f"{campo} debe ser un entero positivo.")
    return valor


def numero(valor, campo):
    try:
        cantidad = Decimal(str(valor))
    except (InvalidOperation, ValueError) as causa:
        # Conserva el motivo técnico de la conversión en el registro de errores.
        raise DatoInvalido(f"{campo} debe ser numérico.") from causa
    if not cantidad.is_finite() or cantidad < 0:
        raise DatoInvalido(f"{campo} debe ser finito y mayor o igual a cero.")
    return cantidad

## 2. Entidades y datos personales
Entidad obliga a cada subclase a describirse. Cliente valida nombre y correo antes del alta y guarda sus datos personales en atributos privados.

In [2]:
class Entidad(ABC):
    def __init__(self, codigo):
        self.__codigo = texto(codigo, "código")

    @property
    def codigo(self):
        return self.__codigo

    @abstractmethod
    def describir(self):
        """Devuelve una descripción legible de la entidad."""


class Cliente(Entidad):
    def __init__(self, codigo, nombre, correo):
        super().__init__(codigo)
        self.__nombre = texto(nombre, "nombre")
        self.__correo = texto(correo, "correo")
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", self.__correo):
            raise DatoInvalido("El correo debe incluir usuario y dominio.")

    def describir(self):
        return f"{self.codigo} · {self.__nombre}"

## 3. Servicios y variantes del cálculo
Una misma llamada a costo elige el subtotal según el tipo de servicio. El descuento se aplica antes del impuesto. Las tasas de ejemplo son parámetros de simulación.

In [3]:
class Servicio(Entidad):
    def __init__(self, codigo, tarifa):
        super().__init__(codigo)
        self.tarifa = numero(tarifa, "tarifa")
        if self.tarifa == 0:
            raise DatoInvalido("La tarifa debe ser mayor que cero.")

    @abstractmethod
    def validar(self, horas):
        """Comprueba las condiciones propias del servicio."""

    @abstractmethod
    def subtotal(self, horas):
        """Calcula el valor antes de ajustes."""

    def costo(self, horas, *, descuento=0, impuesto=0):
        # Las variantes comparten un método con argumentos opcionales.
        self.validar(horas)
        rebaja = numero(descuento, "descuento")
        tasa = numero(impuesto, "impuesto")
        if max(rebaja, tasa) > 100:
            raise DatoInvalido("Los porcentajes deben estar entre 0 y 100.")
        neto = self.subtotal(horas) * (1 - rebaja / 100)
        return (neto * (1 + tasa / 100)).quantize(Decimal("0.01"), ROUND_HALF_UP)


class Sala(Servicio):
    def validar(self, horas):
        if entero(horas, "duración") > 8:
            raise DatoInvalido("Una sala admite hasta 8 horas.")

    def subtotal(self, horas):
        return self.tarifa * horas

    def describir(self):
        return f"Sala {self.codigo}"


class Equipo(Servicio):
    def validar(self, horas):
        if entero(horas, "duración") > 24:
            raise DatoInvalido("Un equipo admite hasta 24 horas.")

    def subtotal(self, horas):
        return self.tarifa * horas + 5000  # Cargo fijo de alistamiento en COP.

    def describir(self):
        return f"Equipo {self.codigo}"


class Asesoria(Servicio):
    def validar(self, horas):
        if entero(horas, "duración") > 4:
            raise DatoInvalido("Una asesoría admite hasta 4 horas.")

    def subtotal(self, horas):
        return self.tarifa * horas + self.tarifa / 2  # Media hora de preparación.

    def describir(self):
        return f"Asesoría {self.codigo}"

## 4. Ciclo de una reserva
Una reserva pasa de pendiente a confirmada. Desde confirmada se puede cancelar o procesar (dar por prestado el servicio). Ambas acciones son definitivas.

In [4]:
class Reserva(Entidad):
    def __init__(self, codigo, cliente, servicio, horas):
        super().__init__(codigo)
        self.cliente = cliente
        self.servicio = servicio
        self.horas = horas
        self.total = servicio.costo(horas)
        self.__estado = "pendiente"

    @property
    def estado(self):
        return self.__estado

    def confirmar(self):
        if self.estado != "pendiente":
            raise OperacionInvalida("Solo se confirma una reserva pendiente.")
        self.__estado = "confirmada"

    def cancelar(self):
        if self.estado != "confirmada":
            raise OperacionInvalida("Solo se cancela una reserva confirmada.")
        self.__estado = "cancelada"

    def procesar(self):
        if self.estado != "confirmada":
            raise OperacionInvalida("Solo se procesa una reserva confirmada.")
        self.__estado = "procesada"

    def describir(self):
        return f"{self.codigo} · {self.servicio.describir()} · {self.total} COP · {self.estado}"

## 5. Coordinación del sistema
Las listas conservan los objetos mientras se ejecuta el cuaderno. Cada servicio admite una sola reserva confirmada simultánea. El ejercicio simula ocupación inmediata, sin calendario.

In [5]:
class Sistema:
    def __init__(self):
        self.clientes = []
        self.servicios = []
        self.reservas = []

    def registrar(self, lista, entidad):
        if any(actual.codigo == entidad.codigo for actual in lista):
            raise OperacionInvalida(f"El código {entidad.codigo} ya existe.")
        lista.append(entidad)
        return entidad.describir()

    def buscar(self, lista, codigo):
        for entidad in lista:
            if entidad.codigo == codigo:
                return entidad
        raise OperacionInvalida(f"No se encontró el código {codigo}.")

    def reservar(self, codigo, cliente_id, servicio_id, horas):
        cliente = self.buscar(self.clientes, cliente_id)
        servicio = self.buscar(self.servicios, servicio_id)
        # Cada código representa un recurso físico o un asesor individual.
        if any(r.servicio is servicio and r.estado == "confirmada" for r in self.reservas):
            raise OperacionInvalida("El servicio está ocupado.")
        reserva = Reserva(codigo, cliente, servicio, horas)
        reserva.confirmar()
        return self.registrar(self.reservas, reserva)

    def cambiar_estado(self, codigo, accion):
        reserva = self.buscar(self.reservas, codigo)
        if accion == "cancelar":
            reserva.cancelar()
        elif accion == "procesar":
            reserva.procesar()
        else:
            raise OperacionInvalida("Acción desconocida.")
        return reserva.describir()

## 6. Simulación reproducible
Las funciones de la tabla se ejecutan una a una. Cada fila declara si debe completarse o qué excepción precisa debe producir. El bloque else registra los éxitos y finally cierra el archivo.

In [6]:
s = Sistema()
# Cada intento indica la excepción esperada. None identifica un caso válido.
casos = [
    ("Registrar cliente", lambda: s.registrar(s.clientes, Cliente("C1", "Ana", "ana@example.org")), None),
    ("Correo inválido", lambda: s.registrar(s.clientes, Cliente("C2", "Luis", "luis")), DatoInvalido),
    ("Cliente repetido", lambda: s.registrar(s.clientes, Cliente("C1", "Ana", "ana@example.org")), OperacionInvalida),
    ("Registrar sala", lambda: s.registrar(s.servicios, Sala("S1", 20000)), None),
    ("Tarifa ilegible", lambda: s.registrar(s.servicios, Equipo("E0", "abc")), DatoInvalido),
    ("Registrar equipo", lambda: s.registrar(s.servicios, Equipo("E1", 8000)), None),
    ("Registrar asesoría", lambda: s.registrar(s.servicios, Asesoria("A1", 40000)), None),
    ("Falta la tarifa", lambda: Sala("S2"), TypeError),
    ("Cliente inexistente", lambda: s.reservar("R0", "C9", "S1", 2), OperacionInvalida),
    ("Reservar sala", lambda: s.reservar("R1", "C1", "S1", 2), None),
    ("Sala ocupada", lambda: s.reservar("R2", "C1", "S1", 1), OperacionInvalida),
    ("Cancelar sala", lambda: s.cambiar_estado("R1", "cancelar"), None),
    ("Procesar tras cancelar", lambda: s.cambiar_estado("R1", "procesar"), OperacionInvalida),
    ("Reutilizar sala liberada", lambda: s.reservar("R2", "C1", "S1", 1), None),
    ("Completar servicio de sala", lambda: s.cambiar_estado("R2", "procesar"), None),
    ("Asesoría demasiado larga", lambda: s.reservar("R3", "C1", "A1", 5), DatoInvalido),
    ("Reservar equipo", lambda: s.reservar("R3", "C1", "E1", 2), None),
    ("Descuento superior al total", lambda: s.buscar(s.servicios, "E1").costo(2, descuento=101), DatoInvalido),
    ("Cotizar equipo con ajustes", lambda: s.buscar(s.servicios, "E1").costo(2, descuento=10, impuesto=19), None),
    ("Reservar asesoría después del error", lambda: s.reservar("R4", "C1", "A1", 2), None),
]

ruta_log = Path("software_fj.log")
registro = logging.getLogger("software_fj.practica")
registro.setLevel(logging.INFO)
registro.propagate = False
salida = logging.FileHandler(ruta_log, mode="w", encoding="utf-8")
salida.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
registro.addHandler(salida)
resultados = []
try:
    for indice, (nombre, operar, esperado) in enumerate(casos, 1):
        try:
            detalle = operar()
        except Exception as error:
            # La frontera de cada operación conserva la traza y permite pasar a la siguiente.
            registro.exception("%02d %s", indice, nombre)
            coincide = esperado is not None and type(error) is esperado
            detalle = f"{type(error).__name__} · {error}"
            estado = "RECHAZADA"
        else:
            coincide = esperado is None
            estado = "ACEPTADA"
            registro.info("%02d %s | %s", indice, nombre, detalle)
        resultados.append(coincide)
        print(f"{indice:02d} {estado} | {nombre} | {detalle}")
finally:
    # Libera el archivo incluso si se interrumpe la simulación.
    registro.removeHandler(salida)
    salida.close()

01 ACEPTADA | Registrar cliente | C1 · Ana
02 RECHAZADA | Correo inválido | DatoInvalido · El correo debe incluir usuario y dominio.
03 RECHAZADA | Cliente repetido | OperacionInvalida · El código C1 ya existe.
04 ACEPTADA | Registrar sala | Sala S1
05 RECHAZADA | Tarifa ilegible | DatoInvalido · tarifa debe ser numérico.
06 ACEPTADA | Registrar equipo | Equipo E1
07 ACEPTADA | Registrar asesoría | Asesoría A1
08 RECHAZADA | Falta la tarifa | TypeError · Servicio.__init__() missing 1 required positional argument: 'tarifa'
09 RECHAZADA | Cliente inexistente | OperacionInvalida · No se encontró el código C9.
10 ACEPTADA | Reservar sala | R1 · Sala S1 · 40000.00 COP · confirmada
11 RECHAZADA | Sala ocupada | OperacionInvalida · El servicio está ocupado.
12 ACEPTADA | Cancelar sala | R1 · Sala S1 · 40000.00 COP · cancelada
13 RECHAZADA | Procesar tras cancelar | OperacionInvalida · Solo se procesa una reserva confirmada.
14 ACEPTADA | Reutilizar sala liberada | R2 · Sala S1 · 20000.00 COP 

## 7. Verificación de los resultados
Las comprobaciones contrastan estados, costos y número de eventos del archivo con lo previsto. Si cambia una regla, revise las expectativas antes de volver a ejecutar.

In [7]:
assert all(resultados), "Un resultado difiere de la simulación prevista."
assert len(s.clientes) == 1 and len(s.servicios) == 3
assert [r.estado for r in s.reservas] == ["cancelada", "procesada", "confirmada", "confirmada"]
assert [r.total for r in s.reservas] == [40000, 20000, 21000, 100000]
assert s.buscar(s.servicios, "E1").costo(2, descuento=10, impuesto=19) == Decimal("22491.00")
bitacora = ruta_log.read_text(encoding="utf-8")
assert bitacora.count(" | ERROR | ") == 9 and bitacora.count(" | INFO | ") == 11
print(f"{sum(resultados)}/{len(casos)} resultados esperados. Registro: {ruta_log.name}")

20/20 resultados esperados. Registro: software_fj.log
